# Assignment 2: English → Hindi Translation with LLM
**Model:** `facebook/nllb-200-distilled-600M` (Meta AI, HuggingFace)

✅ Not Google Translate
✅ Not Helsinki-NLP

> **Run on Google Colab with GPU for best speed**
> Runtime → Change runtime type → GPU

In [ ]:
# ── Cell 1: Install all dependencies ─────────────────────────────────────────
!pip install transformers torch sacrebleu sentencepiece pandas openpyxl -q
print('✅ All packages installed!')

In [ ]:
# ── Cell 2: Upload assignment1_cleaned.csv ────────────────────────────────────
from google.colab import files
print('Please upload assignment1_cleaned.csv')
uploaded = files.upload()

In [ ]:
# ── Cell 3: Load dataset and select 150 sentences ─────────────────────────────
import pandas as pd

df_full = pd.read_csv('assignment1_cleaned.csv')
print(f'Total rows in cleaned dataset: {len(df_full)}')

df = df_full[['English_Sentence', 'Hindi_Sentence']].head(150).copy()
df.reset_index(drop=True, inplace=True)
print(f'Selected {len(df)} sentences for translation')
df.head(3)

In [ ]:
# ── Cell 4: Load NLLB Translation Model (Meta AI) ─────────────────────────────
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = 'facebook/nllb-200-distilled-600M'
print(f'Loading model: {model_name}')
print('This may take 2-3 minutes on first run...')

tokenizer = AutoTokenizer.from_pretrained(model_name)
model     = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = 0 if torch.cuda.is_available() else -1
print(f'Using: {"GPU ✅" if device == 0 else "CPU (slower)"}')

translator = pipeline(
    'translation',
    model=model,
    tokenizer=tokenizer,
    src_lang='eng_Latn',   # English
    tgt_lang='hin_Deva',   # Hindi (Devanagari)
    max_length=512,
    device=device
)
print('✅ Model loaded and ready!')

In [ ]:
# ── Cell 5: Translate 150 English sentences to Hindi ──────────────────────────
translated = []
total = len(df)

for i, sentence in enumerate(df['English_Sentence']):
    try:
        result = translator(str(sentence))
        translated.append(result[0]['translation_text'])
    except Exception as e:
        print(f'  ⚠️ Error at row {i}: {e}')
        translated.append('')
    if (i + 1) % 25 == 0:
        print(f'  Progress: {i+1}/{total} sentences translated...')

df['LLM_Hindi_Translation'] = translated
print(f'\n✅ Translation complete! {len([t for t in translated if t])}/150 successful.')
df[['English_Sentence', 'LLM_Hindi_Translation']].head(5)

In [ ]:
# ── Cell 6: Compute BLEU, CHRF, TER scores ───────────────────────────────────
import sacrebleu

references = df['Hindi_Sentence'].tolist()
hypotheses = df['LLM_Hindi_Translation'].tolist()

bleu = sacrebleu.corpus_bleu(hypotheses, [references])
chrf = sacrebleu.corpus_chrf(hypotheses, [references])
ter  = sacrebleu.corpus_ter(hypotheses,  [references])

bleu_score = round(bleu.score, 4)
chrf_score = round(chrf.score, 4)
ter_score  = round(ter.score,  4)

print('=' * 45)
print('   EVALUATION SCORES')
print('=' * 45)
print(f'  BLEU Score : {bleu_score}   (higher is better)')
print(f'  CHRF Score : {chrf_score}   (higher is better)')
print(f'  TER  Score : {ter_score}  (lower is better)')
print('=' * 45)

In [ ]:
# ── Cell 7: Save scores to scores.txt ────────────────────────────────────────
with open('scores.txt', 'w', encoding='utf-8') as f:
    f.write('=' * 55 + '\n')
    f.write('  Assignment 2 - Translation Evaluation Scores\n')
    f.write('  Model : facebook/nllb-200-distilled-600M (Meta AI)\n')
    f.write('  Task  : English to Hindi Translation\n')
    f.write('  Sentences Evaluated: 150\n')
    f.write('=' * 55 + '\n\n')
    f.write(f'BLEU Score : {bleu_score}\n')
    f.write('  Measures n-gram overlap. Range 0-100. Higher is better.\n\n')
    f.write(f'CHRF Score : {chrf_score}\n')
    f.write('  Character n-gram F-score. Great for Hindi (morphologically rich).\n')
    f.write('  Range 0-100. Higher is better.\n\n')
    f.write(f'TER Score  : {ter_score}\n')
    f.write('  Translation Edit Rate. Edits needed to match reference.\n')
    f.write('  Lower is better.\n\n')
    f.write('=' * 55 + '\n')
    f.write(f'Full BLEU details:\n{bleu}\n\n')
    f.write(f'Full CHRF details:\n{chrf}\n\n')
    f.write(f'Full TER  details:\n{ter}\n')

print('✅ scores.txt saved!')

In [ ]:
# ── Cell 8: Save Excel output ─────────────────────────────────────────────────
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

wb = Workbook()

# ── Sheet 1: Translations ─────────────────────────────────────────────────────
ws1 = wb.active
ws1.title = 'Translations'

header_font  = Font(name='Arial', bold=True, color='FFFFFF', size=11)
header_fill  = PatternFill('solid', start_color='1A5276')
center_align = Alignment(horizontal='center', vertical='center', wrap_text=True)
left_align   = Alignment(horizontal='left',   vertical='center', wrap_text=True)
thin_border  = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'),  bottom=Side(style='thin')
)

headers = ['Original English Sentence', 'Model-Generated Hindi Translation', 'Reference Hindi Sentence']
for col_idx, h in enumerate(headers, start=1):
    cell = ws1.cell(row=1, column=col_idx, value=h)
    cell.font = header_font; cell.fill = header_fill
    cell.alignment = center_align; cell.border = thin_border

fill_light = PatternFill('solid', start_color='D6EAF8')
fill_white  = PatternFill('solid', start_color='FFFFFF')
data_font   = Font(name='Arial', size=10)

for row_idx, row in enumerate(df.itertuples(index=False), start=2):
    fill = fill_light if row_idx % 2 == 0 else fill_white
    vals = [row.English_Sentence, row.LLM_Hindi_Translation, row.Hindi_Sentence]
    for col_idx, val in enumerate(vals, start=1):
        cell = ws1.cell(row=row_idx, column=col_idx, value=val)
        cell.font = data_font; cell.fill = fill
        cell.border = thin_border; cell.alignment = left_align

for i, w in enumerate([60, 60, 60], start=1):
    ws1.column_dimensions[get_column_letter(i)].width = w
ws1.freeze_panes = 'A2'

# ── Sheet 2: Evaluation Scores ────────────────────────────────────────────────
ws2 = wb.create_sheet('Evaluation Scores')
score_headers = ['Metric', 'Score', 'Description', 'Better When']
for col_idx, h in enumerate(score_headers, start=1):
    cell = ws2.cell(row=1, column=col_idx, value=h)
    cell.font = header_font; cell.fill = header_fill
    cell.alignment = center_align; cell.border = thin_border

green_fill = PatternFill('solid', start_color='A9DFBF')
score_data = [
    ['BLEU', bleu_score, 'N-gram precision overlap between hypothesis and reference', 'Higher'],
    ['CHRF', chrf_score, 'Character n-gram F-score, robust for morphologically rich languages like Hindi', 'Higher'],
    ['TER',  ter_score,  'Translation Edit Rate — number of edits needed to match reference', 'Lower'],
]
for row_idx, row_data in enumerate(score_data, start=2):
    for col_idx, val in enumerate(row_data, start=1):
        cell = ws2.cell(row=row_idx, column=col_idx, value=val)
        cell.border = thin_border; cell.alignment = center_align
        if col_idx == 2:
            cell.font = Font(name='Arial', size=12, bold=True)
            cell.fill = green_fill

ws2.column_dimensions['A'].width = 12
ws2.column_dimensions['B'].width = 14
ws2.column_dimensions['C'].width = 68
ws2.column_dimensions['D'].width = 14

meta = [
    ('Model Used:', 'facebook/nllb-200-distilled-600M (Meta AI)'),
    ('Sentences:', 150),
    ('Direction:', 'English to Hindi')
]
for i, (k, v) in enumerate(meta, start=6):
    ws2.cell(row=i, column=1, value=k).font = Font(bold=True, name='Arial')
    ws2.cell(row=i, column=2, value=v)

wb.save('Assignment2_Translation_Output.xlsx')
print('✅ Assignment2_Translation_Output.xlsx saved!')

In [ ]:
# ── Cell 9: Download all output files ────────────────────────────────────────
from google.colab import files
files.download('Assignment2_Translation_Output.xlsx')
files.download('scores.txt')
print('✅ All files downloaded! Upload them to your GitHub repo.')